In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT  = Path(r"G:\FarmWise")
RAW_DIR       = PROJECT_ROOT / "Raw_Data"
PROCESSED_DIR = PROJECT_ROOT / "Processed_Data"
DOC_DIR       = PROJECT_ROOT / "Documentation" / "Data_Sources"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DOC_DIR.mkdir(parents=True, exist_ok=True)

gpm_path = RAW_DIR / "GPM_IMERG" / "GPM_IMERG_Daily_Precipitation_2015_2025.csv"
gpm_out  = PROCESSED_DIR / "GPM_IMERG_Final.csv"
gpm_doc  = DOC_DIR / "GPM_IMERG_Documentation.md"

print("Raw exists:", gpm_path.exists())

Raw exists: True


In [ ]:
with open(gpm_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

header_idx  = next(i for i, l in enumerate(lines) if l.startswith("time,"))
header_text = "".join(lines[:header_idx])
FILL_VALUE  = float(next(l for l in lines if l.startswith("Fill Value")).split(",")[1])

print(header_text)
print("Fill value read from file:", FILL_VALUE)

gpm_raw = pd.read_csv(gpm_path, skiprows=header_idx) 
df = gpm_raw.copy()
df.columns = ["time", "precip_raw"]

print(df.head())
print(df.tail())

User Start Date:,2015-01-01T00:00:00.000Z
User End Date:,2025-09-30T00:00:00.000Z
User Bounding Box:,"31.350000000000005, 30.439999999999998, 31.55, 30.64"
Fill Value:,-9999.900390625
,

Fill value read from file: -9999.900390625
                  time  precip_raw
0  2015-01-01T00:00:00     0.00000
1  2015-01-02T00:00:00     0.00000
2  2015-01-03T00:00:00     0.00000
3  2015-01-04T00:00:00     0.00000
4  2015-01-05T00:00:00     0.02582
                     time  precip_raw
3921  2025-09-26T00:00:00         0.0
3922  2025-09-27T00:00:00         0.0
3923  2025-09-28T00:00:00         0.0
3924  2025-09-29T00:00:00         0.0
3925  2025-09-30T00:00:00         0.0


In [3]:
print("Shape:", df.shape)
print("Dtypes:\n", df.dtypes)

is_fill = np.isclose(df["precip_raw"], FILL_VALUE, atol=0.01)
print("\nNaN values:", df["precip_raw"].isna().sum())
print("Fill-value rows:", is_fill.sum())
print("Any other value < -1000:", ((df["precip_raw"] < -1000) & ~is_fill).sum())
print("Negative values (excluding fill):", ((df["precip_raw"] < 0) & ~is_fill).sum())

valid = df.loc[~is_fill, "precip_raw"]
print("\nTrue zero-rain days:", (valid == 0).sum(), f"({(valid == 0).mean()*100:.1f}% of valid days)")
print("\nDescribe (valid values only):\n", valid.describe())

Shape: (3926, 2)
Dtypes:
 time              str
precip_raw    float64
dtype: object

NaN values: 0
Fill-value rows: 0
Any other value < -1000: 0
Negative values (excluding fill): 0

True zero-rain days: 3412 (86.9% of valid days)

Describe (valid values only):
 count    3926.000000
mean        0.101943
std         1.059557
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        52.054028
Name: precip_raw, dtype: float64


In [4]:
df["date"] = pd.to_datetime(df["time"]).dt.normalize()

expected = pd.date_range("2015-01-01", "2025-09-30", freq="D")

print("First date:", df["date"].min().date(), "| Last date:", df["date"].max().date())
print("Rows:", len(df), "| Expected days:", len(expected))
print("Duplicate dates:", df["date"].duplicated().sum())
print("Sorted ascending:", df["date"].is_monotonic_increasing)

missing_dates = expected.difference(df["date"])
extra_dates   = pd.DatetimeIndex(df["date"]).difference(expected)
print("Missing dates:", len(missing_dates), list(missing_dates[:10]))
print("Unexpected dates:", len(extra_dates))

First date: 2015-01-01 | Last date: 2025-09-30
Rows: 3926 | Expected days: 3926
Duplicate dates: 0
Sorted ascending: True
Missing dates: 0 []
Unexpected dates: 0


In [5]:
d = df.loc[~is_fill].copy()
d["year"]  = d["date"].dt.year
d["month"] = d["date"].dt.month

print("=== Top 15 wettest days (mm/day) ===")
print(d.nlargest(15, "precip_raw")[["date", "precip_raw"]].to_string(index=False))

print("\n=== Yearly totals (mm) and rain days (>=1 mm) ===")
yearly = d.groupby("year")["precip_raw"].agg(total_mm="sum",
                                             rain_days_ge1mm=lambda s: (s >= 1).sum(),
                                             valid_days="count")
print(yearly.round(1))

print("\n=== Mean monthly total across years (mm) ===")
print(d.groupby("month")["precip_raw"].sum().div(d["year"].nunique()).round(1))

print("\n=== Summer days (Jun-Aug) with rain >= 1 mm (to investigate) ===")
summer = d[(d["month"].isin([6, 7, 8])) & (d["precip_raw"] >= 1)]
print(summer[["date", "precip_raw"]].to_string(index=False) if len(summer) else "None")

print("\nValues > 50 mm/day:", (d["precip_raw"] > 50).sum())

=== Top 15 wettest days (mm/day) ===
      date  precip_raw
2020-03-12   52.054028
2020-03-13   15.161750
2019-02-06   13.217810
2020-01-14   12.670126
2016-01-07   12.105085
2018-04-25   10.533967
2020-03-11    9.359044
2020-11-20    8.726250
2018-01-25    7.969394
2015-12-31    7.111553
2021-12-31    6.460958
2022-01-02    6.270091
2015-02-23    5.846436
2023-03-24    5.650026
2022-02-19    5.366569

=== Yearly totals (mm) and rain days (>=1 mm) ===
      total_mm  rain_days_ge1mm  valid_days
year                                       
2015      35.2                8         365
2016      26.1                5         366
2017      16.1                5         365
2018      50.6                9         365
2019      45.8               12         365
2020     129.2               15         366
2021      26.7                6         365
2022      26.3                6         365
2023      29.4                8         365
2024      12.7                4         366
2025       2.4  

In [6]:
gpm_clean = df[["date", "precip_raw"]].copy()

# fill value -> NaN (مش 0!)
gpm_clean.loc[np.isclose(gpm_clean["precip_raw"], FILL_VALUE, atol=0.01), "precip_raw"] = np.nan

gpm_clean = gpm_clean.rename(columns={"precip_raw": "precip_gpm_mm_day"})
gpm_clean["precip_gpm_mm_day"] = gpm_clean["precip_gpm_mm_day"].astype("float64")

gpm_clean = (gpm_clean.sort_values("date")
                      .drop_duplicates(subset="date", keep="first")
                      .reset_index(drop=True))

print("Shape:", gpm_clean.shape)
print("NaN count (former fill values):", gpm_clean["precip_gpm_mm_day"].isna().sum())
print("Dates with NaN:", list(gpm_clean.loc[gpm_clean["precip_gpm_mm_day"].isna(), "date"].dt.date[:20]))
print(gpm_clean.dtypes)
print(gpm_clean.head())

Shape: (3926, 2)
NaN count (former fill values): 0
Dates with NaN: []
date                 datetime64[us]
precip_gpm_mm_day           float64
dtype: object
        date  precip_gpm_mm_day
0 2015-01-01            0.00000
1 2015-01-02            0.00000
2 2015-01-03            0.00000
3 2015-01-04            0.00000
4 2015-01-05            0.02582


In [7]:
import pandas as pd
import numpy as np

print("=== GPM IMERG FINAL VALIDATION ===")

# 1. Expected date range
expected_dates = pd.date_range(
    start="2015-01-01",
    end="2025-09-30",
    freq="D"
)

actual_dates = pd.DatetimeIndex(gpm_clean["date"])

# 2. Date checks
missing_dates = expected_dates.difference(actual_dates)
unexpected_dates = actual_dates.difference(expected_dates)

# 3. Missing values
missing_values = gpm_clean.isna().sum()

# 4. Duplicate dates
duplicate_dates = gpm_clean["date"].duplicated().sum()

# 5. Precipitation checks
negative_values = (
    gpm_clean["precip_gpm_mm_day"] < 0
).sum()

extreme_values = (
    gpm_clean["precip_gpm_mm_day"] > 50
).sum()

# 6. Fill value check
fill_value = -9999.900390625

remaining_fill_values = np.isclose(
    gpm_clean["precip_gpm_mm_day"],
    fill_value,
    atol=0.001
).sum()

# 7. Validation results
checks = {
    "Correct number of records": len(gpm_clean) == 3926,
    "Correct date range": (
        gpm_clean["date"].min() == pd.Timestamp("2015-01-01")
        and
        gpm_clean["date"].max() == pd.Timestamp("2025-09-30")
    ),
    "No missing values": missing_values.sum() == 0,
    "No duplicate dates": duplicate_dates == 0,
    "No missing dates": len(missing_dates) == 0,
    "No unexpected dates": len(unexpected_dates) == 0,
    "Dates sorted": gpm_clean["date"].is_monotonic_increasing,
    "No negative precipitation": negative_values == 0,
    "No remaining fill values": remaining_fill_values == 0
}

print("\n=== DATA CHECKS ===")
print("Shape:", gpm_clean.shape)
print("Missing values:\n", missing_values)
print("Duplicate dates:", duplicate_dates)
print("Missing dates:", len(missing_dates))
print("Unexpected dates:", len(unexpected_dates))
print("Negative precipitation:", negative_values)
print("Remaining fill values:", remaining_fill_values)
print("Values above 50 mm/day:", extreme_values)

print("\n=== VALIDATION SUMMARY ===")

for name, result in checks.items():
    print(f"{name}: {'PASS' if result else 'FAIL'}")

print(
    "\nFINAL RESULT:",
    "ALL CHECKS PASSED" if all(checks.values())
    else "SOME CHECKS FAILED"
)

assert all(checks.values()), "Validation failed. Review the results."

=== GPM IMERG FINAL VALIDATION ===

=== DATA CHECKS ===
Shape: (3926, 2)
Missing values:
 date                 0
precip_gpm_mm_day    0
dtype: int64
Duplicate dates: 0
Missing dates: 0
Unexpected dates: 0
Negative precipitation: 0
Remaining fill values: 0
Values above 50 mm/day: 1

=== VALIDATION SUMMARY ===
Correct number of records: PASS
Correct date range: PASS
No missing values: PASS
No duplicate dates: PASS
No missing dates: PASS
No unexpected dates: PASS
Dates sorted: PASS
No negative precipitation: PASS
No remaining fill values: PASS

FINAL RESULT: ALL CHECKS PASSED


In [8]:
gpm_clean.to_csv(gpm_out, index=False, date_format="%Y-%m-%d")
print("Saved:", gpm_out)

Saved: G:\FarmWise\Processed_Data\GPM_IMERG_Final.csv


In [9]:
chk = pd.read_csv(gpm_out, parse_dates=["date"])
expected = pd.date_range("2015-01-01", "2025-09-30", freq="D")

print("Shape:", chk.shape)
print("First/Last date:", chk["date"].min().date(), "/", chk["date"].max().date())
print("Missing values:", int(chk["precip_gpm_mm_day"].isna().sum()))
print("Duplicate dates:", int(chk["date"].duplicated().sum()))
print("Missing dates:", len(expected.difference(chk["date"])))
print("Contiguous daily:", (chk["date"].diff().dropna() == pd.Timedelta(days=1)).all())
print("Negative values:", int((chk["precip_gpm_mm_day"] < 0).sum()))
print("Values > 300 mm/day:", int((chk["precip_gpm_mm_day"] > 300).sum()))
print("Any leftover fill value:", bool(np.isclose(chk["precip_gpm_mm_day"], FILL_VALUE, atol=0.01).any()))

print("\nFinal statistics:")
print(chk["precip_gpm_mm_day"].describe().round(3))
print("Zero-rain days:", int((chk["precip_gpm_mm_day"] == 0).sum()))

Shape: (3926, 2)
First/Last date: 2015-01-01 / 2025-09-30
Missing values: 0
Duplicate dates: 0
Missing dates: 0
Contiguous daily: True
Negative values: 0
Values > 300 mm/day: 0
Any leftover fill value: False

Final statistics:
count    3926.000
mean        0.102
std         1.060
min         0.000
25%         0.000
50%         0.000
75%         0.000
max        52.054
Name: precip_gpm_mm_day, dtype: float64
Zero-rain days: 3412


In [10]:
def df_to_md(d):
    d = d.reset_index()
    head = "| " + " | ".join(map(str, d.columns)) + " |"
    sep  = "|" + "|".join(["---"] * len(d.columns)) + "|"
    rows = ["| " + " | ".join(map(str, r)) + " |" for r in d.values.tolist()]
    return "\n".join([head, sep] + rows)

n_fill = int(is_fill.sum())
stats_md = df_to_md(chk[["precip_gpm_mm_day"]].describe().T.round(3))
top5 = chk.nlargest(5, "precip_gpm_mm_day").assign(date=lambda x: x["date"].dt.date)
top5_md = df_to_md(top5.set_index("date"))

doc = f"""# GPM IMERG - Daily Precipitation Documentation

## 1. Data Source
- **Provider:** NASA GES DISC
- **Product:** GPM_3IMERGDF (IMERG Final Run, Daily), **Version V07**
- **Spatial resolution:** 0.1°
- **Bounding box (W, S, E, N):** 31.35, 30.44, 31.55, 30.64
- **Raw file:** `Raw_Data/GPM_IMERG_Daily_Precipitation_2015_2025.csv` (never modified)
- **Fill value in source:** {FILL_VALUE}

## 2. Study Area and Period
- **Area:** El-Zankalon, Sharqia, Egypt
- **Period:** 2015-01-01 to 2025-09-30 ({len(chk)} daily records)
- **Note:** GPM ends on 2025-09-30, while NASA POWER ends on 2025-12-31.

## 3. Variables and Units
| Final column | Original name | Description | Unit |
|---|---|---|---|
| date | time | Calendar date (YYYY-MM-DD) | - |
| precip_gpm_mm_day | mean_GPM_3IMERGDF_07_precipitation | Area-averaged daily precipitation over the bounding box | mm/day |

Original units were kept.

## 4. Cleaning Steps
1. Parsed the metadata header and read the fill value from the file.
2. Converted `time` (ISO 8601) to a plain date (YYYY-MM-DD).
3. Replaced the fill value with NaN (fill-value rows found: {n_fill}). Fill values were **not** converted to zero.
4. True zero-rain days were preserved as 0.
5. Sorted by date, checked duplicates and missing dates.
6. No interpolation or imputation was applied.

## 5. Quality Rules
- Precipitation must be >= 0 mm/day.
- Values above 300 mm/day would be flagged as physically implausible for this region.
- Top values and summer (Jun-Aug) rain days were reviewed manually and **kept** (no deletion).
- Missing (NaN) is different from zero precipitation.

## 6. Final Dataset
- **Output file:** `Processed_Data/GPM_IMERG_Final.csv`
- **Shape:** {chk.shape[0]} rows x {chk.shape[1]} columns
- **First date:** {chk['date'].min().date()} | **Last date:** {chk['date'].max().date()}
- **Missing values:** {int(chk['precip_gpm_mm_day'].isna().sum())}
- **Duplicate dates:** {int(chk['date'].duplicated().sum())}
- **Zero-rain days:** {int((chk['precip_gpm_mm_day'] == 0).sum())}

### Final statistics
{stats_md}

### Five wettest days
{top5_md}

## 7. Notes
- IMERG is a satellite-based area average; it is not a point measurement.
- NASA POWER's `precip_mm_day` (MERRA-2 model) is a separate source; both are kept for later comparison.
- Integration with MODIS NDVI must use the overlapping date range only.
"""

gpm_doc.write_text(doc, encoding="utf-8")
print("Documentation saved:", gpm_doc)

Documentation saved: G:\FarmWise\Documentation\Data_Sources\GPM_IMERG_Documentation.md
